# 02. Praproses NLP dan ekstraksi skill (Fase 2)

**Tujuan:** mengubah teks lowongan dan kueri yang tidak terstruktur menjadi token yang konsisten, lalu
mengenali skill di dalamnya dan menyatukan variasi penulisannya ke satu nama baku.

**Yang dipelajari:** lowercase dan cleaning, tokenisasi, stopword (dan kenapa negasi tidak boleh dibuang),
stemming, ekstraksi skill berbasis kamus, normalisasi alias, serta cara mengukur akurasi ekstraksi.

Kode inti ada di `nlp_engine/preprocessing.py` dan `nlp_engine/skills.py`. Buka file itu sambil membaca notebook ini.

In [1]:
import sys, warnings
sys.path.insert(0, "..")          # agar folder nlp_engine/ bisa diimpor
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from nlp_engine import *
from nlp_engine.preprocessing import normalize_text, tokenize, stem, STOPWORDS
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 160)
jobs, queries, qrels, ex = load_data("..")
job_ids = jobs.job_id.tolist()
print(f"{len(jobs)} lowongan, {len(queries)} kueri uji, {len(ex.names)} skill di kamus")

368 lowongan, 30 kueri uji, 56 skill di kamus


## 1. Normalisasi teks
Contoh dari rancangan proyek. Perhatikan apa yang hilang dan apa yang bertahan.

In [2]:
raw = """We are looking for a DATA ANALYST
with experience in Python, SQL,
Pandas, and Power BI!!!"""
print(repr(normalize_text(raw)))
print(repr(normalize_text("Menguasai CI/CD, Node.js, dan C++ (min. 2 tahun)")))   # tanda yang bermakna dipertahankan

'we are looking for a data analyst with experience in python sql pandas and power bi'
'menguasai ci/cd node.js dan c++ min 2 tahun'


## 2. Stopword, dan kenapa negasi dipertahankan
Kata penghubung seperti *yang*, *dan*, *the* jarang membantu pencarian. Tetapi *tanpa*, *bukan*, *not* bisa membalik makna,
jadi daftar stopword proyek ini sengaja tidak memuatnya.

In [3]:
t = normalize_text("Kami mencari analis yang menguasai SQL tanpa pengalaman Python")
print("dengan stopword dibuang :", tokenize(t, do_stem=False))
print("negasi ikut terbuang?   :", any(n in STOPWORDS for n in ["tanpa", "bukan", "tidak", "not", "without"]))

dengan stopword dibuang : ['analis', 'menguasai', 'sql', 'tanpa', 'pengalaman', 'python']
negasi ikut terbuang?   : False


## 3. Stemming ringan
Bahasa Indonesia kaya imbuhan: *mengolah*, *pengolahan*, *diolah* berasal dari *olah*. Stemmer di proyek ini hanya
aturan sederhana. Ia sering benar, kadang keliru (*pelanggan* dipotong terlalu jauh, *menguasai* menjadi *uasai* karena akarnya *kuasa*), dan itu wajar untuk titik awal.
Selama kata yang sama di dokumen dan kueri dipotong dengan cara yang sama, kesalahan konsisten seperti ini tidak merusak pencarian. Yang berbahaya adalah ketidakkonsistenan.

In [4]:
for w in ["mengolah", "pengolahan", "menganalisis", "penjualan", "laporan", "berpengalaman", "pemasaran", "pelanggan", "menguasai", "digital", "dashboard"]:
    print(f"{w:15s} -> {stem(w)}")

mengolah        -> olah
pengolahan      -> olah
menganalisis    -> analisis
penjualan       -> jual
laporan         -> lapor
berpengalaman   -> pengalam
pemasaran       -> asar
pelanggan       -> langg
menguasai       -> uasai
digital         -> digital
dashboard       -> dashboard


> **Latihan 1.** Pasang Sastrawi (`pip install PySastrawi`), ganti fungsi `stem` dengan stemmer Sastrawi,
> lalu bandingkan hasil evaluasi di notebook 03. Apakah stemming yang lebih akurat membuat hasil pencarian lebih baik?

## 4. Ekstraksi skill dan normalisasi alias
`SkillExtractor` membaca `dataset/skills.csv` (nama baku + alias) dan mencocokkan alias pada teks. Alias terpanjang
dicocokkan lebih dulu supaya "power bi desktop" tidak terpecah.

In [5]:
kal = "Saya menguasai MS Excel, Postgre SQL, PowerBI, dan sedikit py untuk mengolah data. Pernah membuat pipeline data juga."
print(ex.extract(kal))
print()
print(ex.canonicalize(normalize_text(kal)))

['Microsoft Excel', 'PostgreSQL', 'Power BI', 'Analisis Data', 'Python', 'ETL']

saya menguasai  sk_microsoft_excel   sk_postgresql   sk_power_bi  dan sedikit  sk_python  untuk  sk_analisis_data  pernah membuat  sk_etl  juga


In [6]:
# tabel normalisasi: variasi tulisan -> nama baku
rows = []
for s in ["MS Excel", "excel", "spreadsheet", "Postgre SQL", "postgres", "PowerBI", "py", "pemasaran digital", "media sosial", "natural language processing"]:
    rows.append({"teks": s, "skill baku": ", ".join(ex.extract(s)) or "-"})
pd.DataFrame(rows)

,teks,skill baku
0,MS Excel,Microsoft Excel
1,excel,Microsoft Excel
2,spreadsheet,Microsoft Excel
3,Postgre SQL,PostgreSQL
4,postgres,PostgreSQL
5,PowerBI,Power BI
6,py,Python
7,pemasaran digital,Digital Marketing
8,media sosial,Social Media
9,natural language processing,NLP


## 5. Seberapa akurat ekstraksinya?
Dataset punya label skill (`required_skills` + `preferred_skills`), sedangkan deskripsi menyebut skill dengan variasi
tulisan. Maka kita bisa mengukur precision, recall, dan F1 ekstraksi dari deskripsi.

(Catatan: label ini dibuat oleh generator yang sama dengan deskripsinya, jadi angkanya cenderung lebih optimistis
daripada pada teks lowongan nyata. Ukur ulang dengan sampel yang diberi label manual.)

In [7]:
from collections import Counter
tp = fp = fn = 0
fp_c, fn_c = Counter(), Counter()
for r in jobs.itertuples():
    gold = set(r.required_skills.split(";")) | (set(r.preferred_skills.split(";")) if isinstance(r.preferred_skills, str) else set())
    pred = set(ex.extract(r.description))
    tp += len(gold & pred); fp += len(pred - gold); fn += len(gold - pred)
    fp_c.update(pred - gold); fn_c.update(gold - pred)
P, R = tp / (tp + fp), tp / (tp + fn)
print(f"Precision {P:.3f}  Recall {R:.3f}  F1 {2*P*R/(P+R):.3f}   (target PRD: >= 0.85)")

Precision 0.922  Recall 0.950  F1 0.936   (target PRD: >= 0.85)


In [8]:
print("Skill yang paling sering salah terbaca (false positive):", fp_c.most_common(6))
print("Skill yang paling sering terlewat (false negative)    :", fn_c.most_common(6))

Skill yang paling sering salah terbaca (false positive): [('Komunikasi', 35), ('Statistik', 19), ('REST API', 11), ('Akuntansi', 9), ('Social Media', 8), ('AWS', 6)]
Skill yang paling sering terlewat (false negative)    : [('SQL', 6), ('Python', 5), ('Microsoft Excel', 5), ('Docker', 4), ('Administrasi', 4), ('Pajak', 4)]


False negative biasanya skill yang ada di label tetapi tidak disebut di deskripsi (lowongan nyata juga begitu: persyaratan
di formulir tidak selalu diulang di teks). False positive biasanya skill yang disebut sebagai bagian lain kalimat
atau alias yang terlalu umum.

> **Latihan 2.** Lihat satu per satu lowongan yang salah dan tentukan: perlu alias baru, atau memang label datanya yang tidak lengkap?
> Tambahkan alias ke `dataset/skills.csv`, jalankan ulang bagian 5, dan catat perubahan F1.

> **Latihan 3.** Dictionary-based tidak mengenali skill yang tidak ada di kamus. Coba rancang aturan regex untuk menangkap
> pola seperti "menguasai X, Y, dan Z" sebagai kandidat skill baru yang perlu ditinjau admin.